# 🚛 Notebook d'Optimisation VRP (Vehicle Routing Problem)

Ce notebook permet d'explorer interactivement le système d'optimisation de tournées de livraison.

## 📋 Contenu
1. Chargement des données
2. Analyse exploratoire
3. Configuration de l'optimisation
4. Résolution du problème
5. Analyse des résultats
6. Visualisations
7. Expérimentations

## 1. Initialisation et Imports

In [ ]:
# Imports nécessaires
import sys
import os

# Ajouter le dossier src au path
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

# Imports des modules VRP
from src.data_handler import DataHandler, get_default_clients_data
from src.optimizer import VRPOptimizer
from src.visualizer import VRPVisualizer
from src.analyzer import VRPAnalyzer

# Imports standards
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration matplotlib
%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 8)
sns.set_style('whitegrid')

print("✓ Imports réussis!")

## 2. Chargement des Données

In [ ]:
# Charger les données par défaut
clients_data = get_default_clients_data()

# Initialiser le gestionnaire de données
data_handler = DataHandler()
clients_df = data_handler.load_clients_data(clients_data)

# Afficher les premières lignes
print("\n📊 Aperçu des données :")
clients_df.head(10)

## 3. Analyse Exploratoire des Données

In [ ]:
# Statistiques descriptives
print("📈 Statistiques descriptives :")
clients_df.describe()

In [ ]:
# Analyser la distribution des demandes
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Distribution des demandes
axes[0, 0].hist(clients_df[clients_df['id'] > 0]['demande'], bins=10, color='skyblue', edgecolor='black')
axes[0, 0].set_title('Distribution des Demandes', fontsize=14, weight='bold')
axes[0, 0].set_xlabel('Demande (unités)')
axes[0, 0].set_ylabel('Nombre de clients')

# Distribution des temps de service
axes[0, 1].hist(clients_df[clients_df['id'] > 0]['temps_service'], bins=10, color='lightcoral', edgecolor='black')
axes[0, 1].set_title('Distribution des Temps de Service', fontsize=14, weight='bold')
axes[0, 1].set_xlabel('Temps de service (min)')
axes[0, 1].set_ylabel('Nombre de clients')

# Répartition des priorités
priority_counts = clients_df[clients_df['id'] > 0]['priorite'].value_counts().sort_index()
axes[1, 0].bar(priority_counts.index, priority_counts.values, color=['gold', 'orange', 'lightblue'], edgecolor='black')
axes[1, 0].set_title('Répartition des Priorités', fontsize=14, weight='bold')
axes[1, 0].set_xlabel('Priorité')
axes[1, 0].set_ylabel('Nombre de clients')
axes[1, 0].set_xticks([1, 2, 3])
axes[1, 0].set_xticklabels(['Haute', 'Moyenne', 'Basse'])

# Distribution spatiale
depot = clients_df[clients_df['id'] == 0]
clients = clients_df[clients_df['id'] > 0]
axes[1, 1].scatter(depot['x'], depot['y'], c='red', s=400, marker='s', label='Dépôt', edgecolors='black', linewidths=2)
axes[1, 1].scatter(clients['x'], clients['y'], c='skyblue', s=100, marker='o', label='Clients', edgecolors='blue')
axes[1, 1].set_title('Distribution Géographique', fontsize=14, weight='bold')
axes[1, 1].set_xlabel('Coordonnée X')
axes[1, 1].set_ylabel('Coordonnée Y')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ Analyse exploratoire terminée")

## 4. Calcul de la Matrice de Distances

In [ ]:
# Calculer la matrice de distances
distance_matrix = data_handler.calculate_distance_matrix()

# Visualiser la matrice de distances
plt.figure(figsize=(12, 10))
sns.heatmap(distance_matrix, cmap='YlOrRd', annot=False, fmt='.1f', 
            xticklabels=clients_df['nom'], yticklabels=clients_df['nom'])
plt.title('Matrice des Distances Euclidiennes', fontsize=16, weight='bold', pad=20)
plt.xlabel('Destination')
plt.ylabel('Origine')
plt.tight_layout()
plt.show()

print(f"\n📏 Distance moyenne : {distance_matrix[distance_matrix > 0].mean():.2f} unités")
print(f"📏 Distance maximale : {distance_matrix.max():.2f} unités")

## 5. Configuration de l'Optimisation

In [ ]:
# Configuration par défaut
config = {
    'num_vehicles': 3,
    'vehicle_capacity': 50,
    'weight_distance': 1.0,
    'weight_priority': 0.5,
    'weight_time': 0.3,
    'time_limit': 30,
    'verbose': False
}

print("⚙️  Configuration :")
for key, value in config.items():
    print(f"  • {key}: {value}")

## 6. Résolution du Problème VRP

In [ ]:
# Créer l'optimiseur
optimizer = VRPOptimizer(data_handler, config)

# Résoudre le problème
print("🔍 Résolution du problème VRP en cours...\n")
solution = optimizer.solve()

if solution:
    print("\n✓ Solution trouvée!")
    # Afficher la solution
    optimizer.print_solution(solution)
else:
    print("\n❌ Aucune solution trouvée")

## 7. Analyse Détaillée des Résultats

In [ ]:
# Créer l'analyseur
analyzer = VRPAnalyzer(data_handler, solution, config)

# Calculer les métriques
metrics = analyzer.calculate_metrics()

# Afficher l'analyse
analyzer.print_analysis()
analyzer.print_comparison()
analyzer.print_recommendations()

In [ ]:
# Créer un DataFrame avec les statistiques par véhicule
vehicle_stats = pd.DataFrame(solution['vehicle_stats'])
vehicle_stats['vehicle_id'] = vehicle_stats['vehicle_id'] + 1

print("\n📊 Tableau récapitulatif par véhicule :")
display(vehicle_stats[['vehicle_id', 'distance', 'load', 'capacity_used']].rename(columns={
    'vehicle_id': 'Véhicule',
    'distance': 'Distance (unités)',
    'load': 'Charge (unités)',
    'capacity_used': 'Utilisation (%)'
}))

## 8. Visualisations

In [ ]:
# Créer le visualiseur
visualizer = VRPVisualizer(data_handler)

# Visualisation des routes
print("\n📊 Visualisation des tournées optimisées :")
visualizer.plot_routes(solution, save_path=None, show=True)

In [ ]:
# Comparaison des distances et charges
print("\n📊 Comparaison des performances par véhicule :")
visualizer.plot_distance_comparison(solution, save_path=None, show=True)

In [ ]:
# Chronologie des tournées
print("\n📊 Chronologie des livraisons :")
visualizer.plot_timeline(solution, save_path=None, show=True)

## 9. Expérimentations - Impact des Paramètres

### 9.1 Impact du Nombre de Véhicules

In [ ]:
# Tester différents nombres de véhicules
vehicle_counts = [2, 3, 4, 5]
results_vehicles = []

print("🧪 Test de l'impact du nombre de véhicules...\n")

for num_vehicles in vehicle_counts:
    config_test = config.copy()
    config_test['num_vehicles'] = num_vehicles
    config_test['verbose'] = False
    
    optimizer_test = VRPOptimizer(data_handler, config_test)
    solution_test = optimizer_test.solve()
    
    if solution_test:
        results_vehicles.append({
            'num_vehicles': num_vehicles,
            'total_distance': solution_test['total_distance'],
            'vehicles_used': len(solution_test['vehicle_stats'])
        })
        print(f"  ✓ {num_vehicles} véhicules : Distance = {solution_test['total_distance']:.2f} unités")

# Visualiser les résultats
df_vehicles = pd.DataFrame(results_vehicles)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(df_vehicles['num_vehicles'], df_vehicles['total_distance'], marker='o', linewidth=2, markersize=10)
ax1.set_xlabel('Nombre de véhicules disponibles', fontsize=12)
ax1.set_ylabel('Distance totale (unités)', fontsize=12)
ax1.set_title('Impact du nombre de véhicules sur la distance', fontsize=14, weight='bold')
ax1.grid(True, alpha=0.3)

ax2.bar(df_vehicles['num_vehicles'], df_vehicles['vehicles_used'], color='skyblue', edgecolor='black')
ax2.set_xlabel('Véhicules disponibles', fontsize=12)
ax2.set_ylabel('Véhicules utilisés', fontsize=12)
ax2.set_title('Véhicules réellement utilisés', fontsize=14, weight='bold')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

### 9.2 Impact de la Capacité des Véhicules

In [ ]:
# Tester différentes capacités
capacities = [40, 50, 60, 70]
results_capacity = []

print("\n🧪 Test de l'impact de la capacité des véhicules...\n")

for capacity in capacities:
    config_test = config.copy()
    config_test['vehicle_capacity'] = capacity
    config_test['verbose'] = False
    
    optimizer_test = VRPOptimizer(data_handler, config_test)
    solution_test = optimizer_test.solve()
    
    if solution_test:
        analyzer_test = VRPAnalyzer(data_handler, solution_test, config_test)
        metrics_test = analyzer_test.calculate_metrics()
        
        results_capacity.append({
            'capacity': capacity,
            'total_distance': solution_test['total_distance'],
            'avg_utilization': metrics_test['avg_capacity_used']
        })
        print(f"  ✓ Capacité {capacity} : Distance = {solution_test['total_distance']:.2f}, Utilisation = {metrics_test['avg_capacity_used']:.1f}%")

# Visualiser les résultats
df_capacity = pd.DataFrame(results_capacity)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(df_capacity['capacity'], df_capacity['total_distance'], marker='s', linewidth=2, markersize=10, color='green')
ax1.set_xlabel('Capacité des véhicules (unités)', fontsize=12)
ax1.set_ylabel('Distance totale (unités)', fontsize=12)
ax1.set_title('Impact de la capacité sur la distance', fontsize=14, weight='bold')
ax1.grid(True, alpha=0.3)

ax2.plot(df_capacity['capacity'], df_capacity['avg_utilization'], marker='s', linewidth=2, markersize=10, color='orange')
ax2.set_xlabel('Capacité des véhicules (unités)', fontsize=12)
ax2.set_ylabel('Taux d\'utilisation moyen (%)', fontsize=12)
ax2.set_title('Impact de la capacité sur l\'utilisation', fontsize=14, weight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 9.3 Impact des Poids d'Optimisation

In [ ]:
# Tester différentes configurations de poids
weight_configs = [
    {'name': 'Distance prioritaire', 'weight_distance': 1.0, 'weight_priority': 0.3, 'weight_time': 0.2},
    {'name': 'Équilibré', 'weight_distance': 1.0, 'weight_priority': 0.5, 'weight_time': 0.3},
    {'name': 'Priorité importante', 'weight_distance': 0.7, 'weight_priority': 1.0, 'weight_time': 0.3},
]

results_weights = []

print("\n🧪 Test de l'impact des poids d'optimisation...\n")

for wconfig in weight_configs:
    config_test = config.copy()
    config_test['weight_distance'] = wconfig['weight_distance']
    config_test['weight_priority'] = wconfig['weight_priority']
    config_test['weight_time'] = wconfig['weight_time']
    config_test['verbose'] = False
    
    optimizer_test = VRPOptimizer(data_handler, config_test)
    solution_test = optimizer_test.solve()
    
    if solution_test:
        analyzer_test = VRPAnalyzer(data_handler, solution_test, config_test)
        metrics_test = analyzer_test.calculate_metrics()
        
        results_weights.append({
            'config': wconfig['name'],
            'distance': solution_test['total_distance'],
            'high_priority_rate': metrics_test['priority_stats']['high_priority']['rate']
        })
        print(f"  ✓ {wconfig['name']} : Distance = {solution_test['total_distance']:.2f}, " +
              f"Haute priorité servie = {metrics_test['priority_stats']['high_priority']['rate']:.1f}%")

# Visualiser les résultats
df_weights = pd.DataFrame(results_weights)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.bar(range(len(df_weights)), df_weights['distance'], color=['#3498db', '#2ecc71', '#e74c3c'], edgecolor='black')
ax1.set_xticks(range(len(df_weights)))
ax1.set_xticklabels(df_weights['config'], rotation=15)
ax1.set_ylabel('Distance totale (unités)', fontsize=12)
ax1.set_title('Impact sur la distance totale', fontsize=14, weight='bold')
ax1.grid(True, alpha=0.3, axis='y')

ax2.bar(range(len(df_weights)), df_weights['high_priority_rate'], color=['#3498db', '#2ecc71', '#e74c3c'], edgecolor='black')
ax2.set_xticks(range(len(df_weights)))
ax2.set_xticklabels(df_weights['config'], rotation=15)
ax2.set_ylabel('Taux de service priorité haute (%)', fontsize=12)
ax2.set_title('Impact sur les clients haute priorité', fontsize=14, weight='bold')
ax2.axhline(y=100, color='red', linestyle='--', linewidth=2, label='100%')
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 10. Export des Résultats

In [ ]:
# Sauvegarder les visualisations
print("💾 Sauvegarde des visualisations...")
visualizer.create_all_visualizations(solution, output_dir='output')

# Exporter l'analyse
analyzer.export_results(output_dir='output', filename='vrp_analysis.json')

# Exporter les données
data_handler.export_to_csv('output/clients_data.csv')

print("\n✓ Tous les résultats ont été sauvegardés dans le dossier 'output/'")

## 11. Conclusion

Ce notebook a permis d'explorer le système d'optimisation VRP et de :

1. ✅ Charger et analyser les données des clients
2. ✅ Résoudre le problème d'optimisation avec OR-Tools
3. ✅ Analyser les résultats et les métriques
4. ✅ Visualiser les tournées optimisées
5. ✅ Expérimenter avec différents paramètres

### Prochaines Étapes

- Tester avec vos propres données
- Ajuster les paramètres selon vos besoins
- Utiliser l'API Flask pour l'automatisation
- Configurer le workflow n8n pour la production

### Ressources

- Documentation : `docs/`
- API Flask : `python main.py api`
- Workflow n8n : `n8n/workflow_vrp.json`